# Module 4 - Class 2: Customer Churn Prediction
**Khamidullokhon Abduvokhidov**

In [ ]:
# Load, clean, one-hot encode, split, and scale Telco customer data.
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_curve, roc_auc_score, classification_report, ConfusionMatrixDisplay
url = 'https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv'
df = pd.read_csv(url)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce').fillna(df['TotalCharges'].median())
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})
cat_cols = df.select_dtypes(include='object').columns.drop('customerID')
df_encoded = pd.get_dummies(df.drop('customerID', axis=1), columns=cat_cols, drop_first=True)
X, y = df_encoded.drop('Churn', axis=1), df_encoded['Churn']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
scaler = StandardScaler()
X_train.loc[:, num_cols] = scaler.fit_transform(X_train[num_cols])
X_test.loc[:, num_cols] = scaler.transform(X_test[num_cols])

In [ ]:
# Train logistic regression and show classification diagnostics.
model = LogisticRegression(max_iter=1000).fit(X_train, y_train)
y_pred, y_proba = model.predict(X_test), model.predict_proba(X_test)[:, 1]
print(classification_report(y_test, y_pred, target_names=['No Churn', 'Churn']))
ConfusionMatrixDisplay.from_estimator(model, X_test, y_test, display_labels=['No Churn', 'Churn'])
plt.title('Confusion Matrix'); plt.show()
fpr, tpr, _ = roc_curve(y_test, y_proba); auc = roc_auc_score(y_test, y_proba)
plt.plot(fpr, tpr, label=f'Logistic Regression (AUC = {auc:.3f})'); plt.plot([0,1], [0,1], 'k--', label='Random Classifier')
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate'); plt.title('ROC Curve'); plt.legend(); plt.show()

In [ ]:
# Interpret coefficients and compare business decision thresholds.
coef_df = pd.DataFrame({'Feature': X_train.columns, 'Coefficient': model.coef_[0]}).sort_values('Coefficient')
display(coef_df.head(5)); display(coef_df.tail(5))
rows = []
for threshold in [0.3, 0.5, 0.7]:
    pred = (y_proba >= threshold).astype(int)
    rows.append([threshold, precision_score(y_test, pred), recall_score(y_test, pred), f1_score(y_test, pred)])
display(pd.DataFrame(rows, columns=['Threshold', 'Precision', 'Recall', 'F1']))
print('A lower threshold catches more potential churners (higher recall), but it also contacts more non-churners.')